# Aula Semana 02: CRISP-DM Fase 2 — Entendimento dos Dados, Métricas de Avaliação & Distribuições Estatísticas

## 1. Visão Geral
Nesta aula, aprofundaremos a **Fase 2 do CRISP-DM (Data Understanding)** utilizando o dataset de concessão de crédito (`CRISP_DM_Random_Forest_Credito.ipynb`), disponível na pasta [`materiais/`](../materiais/CRISP_DM_Random_Forest_Credito.ipynb).

Estudaremos:
1. **Métricas de Avaliação de Modelos (Fase 1 & 5)**: Matriz de Confusão, Acurácia, Precisão, Recall, F1-Score, Curva ROC e ROC-AUC.
2. **Distribuições Estatísticas & Geração do Dataset (`np.random`)**: Curva Normal/Gaussiana (Desvio Padrão), Distribuição de Poisson, Exponencial e Uniforme.
3. **Anatomia e Inspeção Prática do Dataset**: Dicionário de 15 atributos, tipos de ruidos, valores ausentes (`NaN`), outliers e desbalanceamento de classe.

---

## 2. Métricas de Avaliação de Classificação & Matriz de Confusao

Para avaliar modelos de Machine Learning em problemas de risco de crédito, utilizamos a **Matriz de Confusão** e métricas derivadas.

![Matriz de Confusão e Métricas](images/confusion_matrix_metrics.jpg)

### Fórmulas Matemáticas das Métricas:

- **Acurácia**: Proporção geral de acertos do modelo.
$$	ext{Acurácia} = rac{TP + TN}{TP + TN + FP + FN}$$

- **Precisão**: Dentre todas as previsões positivas feitas pelo modelo, quantas eram realmente positivas.
$$	ext{Precisão} = rac{TP}{TP + FP}$$

- **Recall (Sensibilidade)**: Dentre todos os casos reais de inadimplência, quantos o modelo conseguiu detectar. **É a métrica chave para crédito!**
$$	ext{Recall} = rac{TP}{TP + FN}$$

- **F1-Score**: Média harmônica entre Precisão e Recall.
$$	ext{F1-Score} = 2 	imes rac{	ext{Precisão} 	imes 	ext{Recall}}{	ext{Precisão} + 	ext{Recall}}$$

---

### Curva ROC e Área sob a Curva (ROC-AUC)

A **Curva ROC** (*Receiver Operating Characteristic*) plota a taxa de Verdadeiros Positivos ($TPR = Recall$) contra a taxa de Falsos Positivos ($FPR = rac{FP}{FP + TN}$) para diferentes limiares de decisão de probabilidade.

![Curva ROC e ROC-AUC](images/roc_curve_auc.jpg)

- **ROC-AUC = 0.5**: Modelo equivalente a um chutar aleatório (moeda).
- **ROC-AUC >= 0.85**: Excelente capacidade de separação estatística entre caloteiros e bons pagadores.


In [ ]:
# Exemplo de calculo das metricas de classificacao e Curva ROC
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

# Simulação de rótulos reais e probabilidades preditas por um modelo
y_real = [0, 0, 0, 0, 0, 0, 1, 1, 1, 1]
y_prob = [0.1, 0.2, 0.15, 0.3, 0.7, 0.25, 0.85, 0.9, 0.65, 0.4]
y_pred = [1 if p >= 0.5 else 0 for p in y_prob]

# Exibindo Matriz de Confusão e Métrica ROC-AUC
cm = confusion_matrix(y_real, y_pred)
auc = roc_auc_score(y_real, y_prob)

print(classification_report(y_real, y_pred, target_names=['Adimplente (0)', 'Inadimplente (1)']))
print(f"Área sob a Curva ROC (ROC-AUC): {auc:.4f}")

# Plot gráfico da Matriz de Confusão
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Prev. 0', 'Prev. 1'], yticklabels=['Real 0', 'Real 1'])
plt.title('Matriz de Confusão Demonstrativa')
plt.show()


---
## 3. Distribuições Estatísticas e Geração de Dados com `np.random`

O dataset da disciplina é gerado via simulação de monte carlo para incorporar padrões estatísticos reais do mercado financeiro.

![Distribuições Estatísticas](images/normal_poisson_distributions.jpg)

### A) Distribuição Normal / Gaussiana (`np.random.normal`)
Utilizada para modelar variáveis contínuas da natureza que se concentram em torno de uma média $\mu$ com dispersão medida pelo desvio padrão $\sigma$.
- **Idade**: `np.random.normal(loc=38.0, scale=11.0)`
- **Score Serasa**: `np.random.normal(loc=620.0, scale=140.0)`

#### Regra Empírica 68-95-99.7:
- **$68\%$** dos dados estão a $\pm 1\sigma$ da média.
- **$95\%$** dos dados estão a $\pm 2\sigma$ da média.
- **$99.7\%$** dos dados estão a $\pm 3\sigma$ da média.

### B) Distribuição de Poisson (`np.random.poisson`)
Utilizada para modelar variáveis discretas que representam contagens de eventos independentes em um intervalo fixo.
- **Consultas recentes ao CPF**: `np.random.poisson(lam=2.3)`
- **Histórico de atrasos em 12 meses**: `np.random.poisson(lam=0.8)`

### C) Outros Recursos Usados no Dataset:
- `np.random.choice()`: Amostragem categórica com probabilidades (ex: `CLT`, `Autonomo`, `PJ`).
- `np.random.exponential()`: Distribuição de renda assimétrica com cauda longa à direita.
- `np.clip()`: Limitação de valores dentro de um intervalo válido (ex: Score de 150 a 990).


In [ ]:
# Simulação visual da geração de variáveis com np.random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# Gerando amostras sintéticas
idades = np.random.normal(loc=38.0, scale=11.0, size=1000)
consultas_poisson = np.random.poisson(lam=2.3, size=1000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico 1: Distribuição Normal de Idades
sns.histplot(idades, kde=True, ax=axes[0], color='royalblue')
axes[0].set_title('Distribuição Normal (Idades: µ=38, σ=11)')
axes[0].set_xlabel('Idade (anos)')

# Gráfico 2: Distribuição de Poisson de Consultas
sns.countplot(x=consultas_poisson, ax=axes[1], palette='crest')
axes[1].set_title('Distribuição de Poisson (Consultas CPF: λ=2.3)')
axes[1].set_xlabel('Número de Consultas')

plt.tight_layout()
plt.show()


---
## 4. Inspeção Prática do Dataset de Crédito com Pandas

Abaixo, executamos os 5 comandos essenciais para diagnóstico da base bruta `df_raw`.


In [ ]:
# Gerando a base sintética idêntica ao notebook de materiais
np.random.seed(42)
n_amostras = 3000

vinculos_brutos = ['CLT', ' CLT ', 'clt', 'Autonomo', 'AUTONOMO', 'PJ', 'Servidor_Publico', 'Estudante']
vinculos = np.random.choice(vinculos_brutos, size=n_amostras, p=[0.35, 0.10, 0.05, 0.20, 0.05, 0.12, 0.08, 0.05])
estados_civis = np.random.choice(['Solteiro', 'Casado', 'Divorciado', 'Viuvo'], size=n_amostras, p=[0.45, 0.40, 0.12, 0.03])
escolaridades = np.random.choice(['Medio', 'Superior', 'Pos_Graduacao', 'Fundamental'], size=n_amostras, p=[0.40, 0.42, 0.12, 0.06])

idade = np.random.normal(loc=38.0, scale=11.0, size=n_amostras)
renda_mensal = np.random.exponential(scale=4200.0, size=n_amostras) + 1400.0
valor_solicitado = np.random.uniform(2000.0, 35000.0, size=n_amostras)
numero_parcelas = np.random.choice([12, 24, 36, 48, 60], size=n_amostras, p=[0.2, 0.3, 0.25, 0.15, 0.10])
score_serasa = np.clip(np.random.normal(loc=620.0, scale=140.0, size=n_amostras), 150.0, 990.0)
num_consultas_recentes = np.random.poisson(lam=2.3, size=n_amostras)
atrasos_ultimos_12m = np.random.poisson(lam=0.8, size=n_amostras)

# Ruidos
idade[np.random.choice(n_amostras, size=4, replace=False)] = 999.0
idade[np.random.choice(n_amostras, size=4, replace=False)] = -15.0
renda_mensal[np.random.choice(n_amostras, size=6, replace=False)] = 999999.0
renda_mensal[np.random.choice(n_amostras, size=int(0.06 * n_amostras), replace=False)] = np.nan
score_serasa[np.random.choice(n_amostras, size=int(0.05 * n_amostras), replace=False)] = np.nan

# Alvo
parcela_mensal = valor_solicitado / numero_parcelas
comprometimento = parcela_mensal / (renda_mensal + 1e-5)
stress = (comprometimento * 4.5) + (atrasos_ultimos_12m * 0.85) + (num_consultas_recentes * 0.25) - ((score_serasa - 600.0) / 130.0) - ((renda_mensal - 4000.0) / 3500.0) + np.random.normal(0.0, 0.9, size=n_amostras)
prob_inad = 1.0 / (1.0 + np.exp(-(stress - 1.8)))
inadimplente = (np.random.rand(n_amostras) < prob_inad).astype(int)

df_raw = pd.DataFrame({
    'proponente_id': [f"PROP-{10000 + i}" for i in range(n_amostras)],
    'idade': np.round(idade, 1),
    'tipo_vinculo': vinculos,
    'estado_civil': estados_civis,
    'escolaridade': escolaridades,
    'renda_mensal': np.round(renda_mensal, 2),
    'valor_solicitado': np.round(valor_solicitado, 2),
    'numero_parcelas': numero_parcelas,
    'score_serasa': np.round(score_serasa, 1),
    'num_consultas_recentes': num_consultas_recentes,
    'atrasos_ultimos_12m': atrasos_ultimos_12m,
    'ruido_estocastico': np.round(np.random.normal(0.0, 50.0, size=n_amostras), 2),
    'numero_da_sorte_app': np.random.randint(1000, 9999, size=n_amostras),
    'ip_origem_hash': [f"IP-{hex(np.random.randint(100000, 999999))[2:].upper()}" for _ in range(n_amostras)],
    'inadimplente': inadimplente
})

print("Shape do dataset bruto:", df_raw.shape)
display(df_raw.head())


In [ ]:
# Comandos de inspeção e diagnóstico inicial
print("--- 1. Informações do DataFrame ---")
df_raw.info()

print("\n--- 2. Contagem de Valores Nulos ---")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

print("\n--- 3. Desbalanceamento da Classe Alvo ---")
print(df_raw['inadimplente'].value_counts(normalize=True) * 100)


---
## 5. Exercícios de Fixação

1. Explique por que o **Recall** é a métrica mais relevante do que a **Acurácia** ao avaliar modelos de concessão de crédito.
2. Na distribuição Normal de idades ($\mu = 38$, $\sigma = 11$), aproximadamente qual percentual de clientes possui idade entre 27 e 49 anos?
3. Qual função do `np.random` foi utilizada para gerar a contagem discreta de atrasos em 12 meses?
4. O que indica uma Curva ROC com área sob a curva (ROC-AUC) igual a 0.50?
